# Chromium 135.0.7049.91 源码拉取（Colab 版）
流程与 Kaggle kernel 相同：拉取 costa-core 基准版本 135 + 工具链，流式分卷输出到 /content/drive 供国内机器下载。
运行方式：Runtime → Run all。总时长约 90-150 分钟。

In [ ]:
import os, subprocess, sys, time, shutil

def run(cmd, cwd=None, timeout=None, check=True):
    print(f"\n$ {cmd}", flush=True)
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, cwd=cwd, timeout=timeout, executable="/bin/bash")
    print(f"  -> exit={r.returncode} ({time.time()-t0:.0f}s)", flush=True)
    if check and r.returncode != 0:
        sys.exit(f"step failed: {cmd}")

def run_retry(cmd, tries=5, **kw):
    for i in range(1, tries+1):
        if run(cmd, check=False, **kw) == 0:
            return
        print(f"retry {i}/{tries} after 20s...", flush=True)
        time.sleep(20)
    sys.exit(f"failed after {tries}: {cmd}")

# 挂 Google Drive（产物存放，之后从国内经 Drive/其他通道取回）
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/chromium-135'
os.makedirs(WORK + '/bundle', exist_ok=True)
print('WORK =', WORK)

In [ ]:
SRC_ROOT = '/content/chromium'
os.makedirs(SRC_ROOT, exist_ok=True)
os.environ['DEPOT_TOOLS_METRICS'] = '0'

run("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq git curl lsb-release file zstd ca-certificates > /dev/null")
run_retry("git clone --depth 1 https://chromium.googlesource.com/chromium/tools/depot_tools.git /content/depot_tools", tries=6)
os.environ['PATH'] = '/content/depot_tools:' + os.environ['PATH']
run_retry('gclient --version', tries=6)

In [ ]:
# 单阶段 pin 135.0.7049.91（.gclient 直接锁版本，避免双倍拉取）
with open(os.path.join(SRC_ROOT, '.gclient'), 'w') as f:
    f.write(
        'solutions = [\n'
        '  {"name": "src", "url": "https://chromium.googlesource.com/chromium/src.git@135.0.7049.91",\n'
        '   "managed": False, "custom_deps": {}, "custom_vars": {}, "deps_file": ".DEPS.git"}\n'
        ']\n')
run_retry('gclient sync --shallow --with_branch_heads --nohooks', cwd=SRC_ROOT, timeout=6*3600, tries=4)
print('SYNC DONE', flush=True)

In [ ]:
SRC = os.path.join(SRC_ROOT, 'src')
run('./build/install-build-deps.sh --no-arm --no-chromeos-fonts --no-prompt', cwd=SRC, timeout=3600)
run_retry('gclient runhooks', cwd=SRC, timeout=4*3600, tries=3)
run(f"git -C src rev-parse HEAD > {WORK}/REVISION.txt", cwd=SRC_ROOT)
run("echo 135.0.7049.91 >> " + WORK + "/REVISION.txt", cwd=SRC_ROOT)
print('HOOKS DONE', flush=True)

In [ ]:
# 流式分卷打包到 Drive
run(f"tar -cf - -C {SRC_ROOT} src | zstd -T0 -15 | split -b 900M -d --additional-suffix=.part - {WORK}/bundle/src.part.", timeout=6*3600)
run(f"tar -cf - -C /content depot_tools | zstd -T0 -15 -o {WORK}/bundle/depot_tools.tar.zst", timeout=600)
run(f"cd {WORK}/bundle && sha256sum src.part.* depot_tools.tar.zst > SHA256SUMS && ls -lh | head -20")
print('=== ALL DONE: bundle in Drive chromium-135/bundle ===', flush=True)